In [1]:
!pip install transformers datasets rouge_score sentencepiece nltk scikit-learn accelerate -q

  Preparing metadata (setup.py) ... done


In [2]:
import numpy as np
import nltk
nltk.download("punkt")

from nltk.tokenize import sent_tokenize

from datasets import load_dataset

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

from rouge_score import rouge_scorer

from transformers import (
    BartTokenizer,
    BartForConditionalGeneration,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments
)

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


In [3]:
dataset = load_dataset("cnn_dailymail", "3.0.0")

train_data = dataset["train"].select(range(5000))
val_data   = dataset["validation"].select(range(500))
test_data  = dataset["test"].select(range(500))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

3.0.0/train-00000-of-00003.parquet:   0%|          | 0.00/257M [00:00<?, ?B/s]

3.0.0/train-00001-of-00003.parquet:   0%|          | 0.00/257M [00:00<?, ?B/s]

3.0.0/train-00002-of-00003.parquet:   0%|          | 0.00/259M [00:00<?, ?B/s]

3.0.0/validation-00000-of-00001.parquet:   0%|          | 0.00/34.7M [00:00<?, ?B/s]

3.0.0/test-00000-of-00001.parquet:   0%|          | 0.00/30.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/287113 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/13368 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/11490 [00:00<?, ? examples/s]

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
model_path = "/content/drive/MyDrive/bart_final"
tokenizer = BartTokenizer.from_pretrained(model_path)
model = BartForConditionalGeneration.from_pretrained(model_path)

Please make sure the generation config includes `forced_bos_token_id=0`. 


Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

In [6]:
oracle_scorer = rouge_scorer.RougeScorer(
    ['rouge2'],
    use_stemmer=True
)

In [7]:
def get_oracle_train(article, reference, n=4):

    sents = sent_tokenize(article)

    if len(sents) <= n:
        return " ".join(sents)

    selected = []
    selected_idx = []

    for _ in range(n):
        best_score = -1
        best_idx = -1

        for i, sent in enumerate(sents):
            if i in selected_idx:
                continue

            candidate = " ".join(selected + [sent])

            score = oracle_scorer.score(
                reference,
                candidate
            )["rouge2"].fmeasure

            if score > best_score:
                best_score = score
                best_idx = i

        if best_idx == -1:
            break

        selected.append(sents[best_idx])
        selected_idx.append(best_idx)

    selected_idx = sorted(selected_idx)

    return " ".join([sents[i] for i in selected_idx])

In [8]:
def get_oracle_test(article, n=4):

    sents = sent_tokenize(article)

    if len(sents) <= n:
        return " ".join(sents)

    vect = TfidfVectorizer(stop_words="english")

    X = vect.fit_transform([article] + sents)

    article_vec = X[0]
    sent_vecs   = X[1:]

    scores = cosine_similarity(article_vec, sent_vecs)[0]

    top_idx = np.argsort(scores)[-n:]
    top_idx = sorted(top_idx)

    return " ".join([sents[i] for i in top_idx])

In [9]:
MAX_INPUT = 1024
MAX_TARGET = 128

In [10]:
def preprocess_train(example):

    oracle = get_oracle_train(
        example["article"],
        example["highlights"],
        n=3
    )

    guided_input = (
         oracle
        + " </s> "
        + example["article"]
    )

    inputs = tokenizer(
        guided_input,
        max_length=MAX_INPUT,
        truncation=True,
        padding="max_length"
    )

    labels = tokenizer(
        example["highlights"],
        max_length=MAX_TARGET,
        truncation=True,
        padding="max_length"
    )
    label_ids = labels["input_ids"]
    label_ids = [-100 if t == tokenizer.pad_token_id else t for t in label_ids]

    inputs["labels"] = label_ids

    return inputs

In [11]:
def preprocess_test(article):

    oracle = get_oracle_test(article, n=3)
    guided_input = (
        oracle + " </s> " + article
    )
    return guided_input

In [12]:
import nltk
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [13]:
tokenized_train = train_data.map(preprocess_train)
tokenized_val   = val_data.map(preprocess_train)

Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

In [14]:
args = Seq2SeqTrainingArguments(

    output_dir="./oracle_bart",

    num_train_epochs=3,

    learning_rate= 2e-5,

    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,
    fp16=True,
    logging_steps=100,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    predict_with_generate=True,

    report_to="none"
)

In [15]:
trainer = Seq2SeqTrainer(
    model=model,
    args=args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    # tokenizer=tokenizer,
    data_collator=DataCollatorForSeq2Seq(tokenizer, model=model)
)

trainer.train()

Epoch,Training Loss,Validation Loss
1,3.600504,1.841942
2,2.601923,2.026713
3,1.750654,2.261286


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['model.encoder.embed_tokens.weight', 'model.decoder.embed_tokens.weight', 'lm_head.weight'].


TrainOutput(global_step=1875, training_loss=2.5216461669921877, metrics={'train_runtime': 2664.6727, 'train_samples_per_second': 5.629, 'train_steps_per_second': 0.704, 'total_flos': 3.250656903168e+16, 'train_loss': 2.5216461669921877, 'epoch': 3.0})

In [16]:
from google.colab import drive
drive.mount('/content/drive')
trainer.save_model("/content/drive/MyDrive/oracle_bart_v2")
tokenizer.save_pretrained("/content/drive/MyDrive/oracle_bart_v2")
print("Saved.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved.


In [17]:
def summarize_oracle(article):

    text = preprocess_test(article)

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_INPUT
    ).to(model.device)

    ids = model.generate(
        inputs["input_ids"],

        num_beams=4,
        length_penalty=2.0,
        no_repeat_ngram_size=3,

        max_length=128,
        min_length=30,

        early_stopping=True
    )

    return tokenizer.decode(
        ids[0],
        skip_special_tokens=True
    )

In [18]:
eval_scorer = rouge_scorer.RougeScorer(
    ['rouge1','rouge2','rougeL'],
    use_stemmer=True
)

r1,r2,rl = [],[],[]

for i in range(200):

    pred = summarize_oracle(test_data[i]["article"])
    ref  = test_data[i]["highlights"]

    score = eval_scorer.score(ref, pred)

    r1.append(score["rouge1"].fmeasure)
    r2.append(score["rouge2"].fmeasure)
    rl.append(score["rougeL"].fmeasure)

print("ROUGE-1:", np.mean(r1))
print("ROUGE-2:", np.mean(r2))
print("ROUGE-L:", np.mean(rl))

ROUGE-1: 0.35516647666595985
ROUGE-2: 0.14459065659688972
ROUGE-L: 0.24097679247452164
